# Playground: talk to `base_100m`

A 100M-parameter Llama-style model, [built and pretrained from scratch](https://github.com/efe-tatar/transformer-pytorch)
on 3B tokens of FineWeb-Edu. This notebook downloads it from the
[Hugging Face Hub](https://huggingface.co/Efe-Tatar/base-100m-fineweb-edu) and runs it on the project's
**own inference code** (its `CausalLM` with KV cache), not on the `transformers` library.

No account or token needed. A **CPU runtime** is enough (Runtime → Change runtime type → CPU); a GPU is faster.

This is a **base model**: it continues text, it doesn't follow instructions.
Start the text you want it to write: `Photosynthesis is the process by which` works much better than
`What is photosynthesis?`.

**Runtime → Run all**, then type in the box at the bottom.

## 1. Code and model (~1 min, the model is 400 MB)

In [ ]:
import os
REPO = "/content/transformer-pytorch"
if not os.path.exists(REPO):
    !git clone -q https://github.com/efe-tatar/transformer-pytorch.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q regex safetensors

import torch
from tools.load_hf import load_hf

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = load_hf("Efe-Tatar/base-100m-fineweb-edu", device=device)
EOT = tokenizer.special_tokens["<|endoftext|>"]
print(f"loaded {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters on {device}")

## 2. Generation

Every training document started with `<|endoftext|>`, so the prompt does too: the model then reads it as the
start of a new text. Tokens are printed as soon as they are sampled.

In [ ]:
def complete(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    ids = torch.tensor([[EOT] + tokenizer.encode(prompt, allowed_special="none")], device=device)
    new, printed = [], ""
    for token in model.stream(ids, max_new_tokens, temperature, top_k, stop_token=EOT):
        if token.item() == EOT:
            break
        new.append(token.item())
        text = tokenizer.decode(new)
        # a multi-byte character can be split across tokens: wait until it's complete
        if not text.endswith("\ufffd"):
            print(text[len(printed):], end="", flush=True)
            printed = text
    print()

print("Photosynthesis is the process by which", end="")
complete("Photosynthesis is the process by which")

## 3. Your turn

Type the beginning of a text, the model continues it. Empty line to stop.

- `TEMPERATURE`: lower (0.5) = safer and more repetitive, higher (1.0) = more creative and more chaotic, 0 = always the most likely token.
- Prompts that work well: `The three main causes of the French Revolution were`, `In this lesson, students will learn`,
  `The human heart has four chambers:`.

In [ ]:
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 200
while True:
    prompt = input("\nyou> ")
    if not prompt.strip():
        break
    print("model> " + prompt, end="")
    complete(prompt, MAX_NEW_TOKENS, TEMPERATURE)